# enriquecimento dos currículos lattes com openalex

este notebook realiza o enriquecimento dos currículos dos egressos a partir dos arquivos xml do lattes.

o notebook não utiliza uma planilha intermediária de orcid.

o fluxo é:

**arquivos xml do lattes → extração das informações do currículo → identificação do orcid → consulta à api do openalex → publicações → indicadores → arquivo excel**

a pasta com os currículos lattes está disponibilizada diretamente no ambiente do google colab.

o objetivo é utilizar o orcid encontrado no próprio currículo lattes como identificador para consultar as publicações associadas ao pesquisador no openalex.

os resultados serão organizados para posterior utilização nos módulos de análise, PLN e BI do projeto ragi.

In [ ]:
!pip -q install pandas openpyxl requests lxml tqdm

In [ ]:
import os
import re
import time
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd
import requests

from tqdm.auto import tqdm
from datetime import datetime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. configuração dos diretórios

os arquivos xml dos currículos lattes devem estar disponíveis no ambiente do colab.

o notebook irá procurar os arquivos na pasta:

`/content/Extrator de currículo - LATTES/Currículos dos Egressos - LATTES`

não é necessário conectar o google drive.

os arquivos xml serão tratados como a fonte de entrada do módulo openalex.

In [ ]:
# diretório principal do projeto
DIRETORIO_PROJETO = "/content/drive/MyDrive/RAGI 2026 2"

# pasta onde estão os currículos xml
DIRETORIO_XML = os.path.join(
    DIRETORIO_PROJETO,
    "Extrator de Currículo - LATTES",
    "Currículos dos Egressos - LATTES"
)

print("procurando a pasta:")
print(DIRETORIO_XML)
print()

if os.path.isdir(DIRETORIO_XML):
    print("✓ pasta encontrada com sucesso!")
else:
    print("✗ pasta não encontrada.")

procurando a pasta:
/content/drive/MyDrive/RAGI 2026 2/Extrator de Currículo - LATTES/Currículos dos Egressos - LATTES

✓ pasta encontrada com sucesso!


In [ ]:
# localizar todos os arquivos xml da pasta

arquivos_xml = sorted(
    [
        os.path.join(DIRETORIO_XML, arquivo)
        for arquivo in os.listdir(DIRETORIO_XML)
        if arquivo.lower().endswith(".xml")
    ]
)

print(f"total de arquivos xml encontrados: {len(arquivos_xml)}")
print()

for arquivo in arquivos_xml:
    print(os.path.basename(arquivo))

total de arquivos xml encontrados: 271

Abdul_Pedro_Manuel_Muchingeca_9666992319712718.xml
Abner_Antônio_Ferreira_0596211008110321.xml
Adriana_Teixeira_de_Moraes_3100726985926660.xml
Adrielly_Campos_e_Almeida_4039893861746107.xml
Aldenor_da_Silva_Pimentel_5770288768881926.xml
Alessandra_Rodrigues_Oliveira_9237912840073975.xml
Alessandra_Siqueira_Lessa_5616998500202010.xml
Aline_Dos_Santos_7369015309091344.xml
Amanda_Costa_e_Silva_5421893766331754.xml
Amanda_de_Sousa_Veloso_2847738122151795.xml
Ana_Clara_Gomes_Costa_1160526089881609.xml
Ana_Domitila_Rosa_Lemos_Silva_5548471284689159.xml
Ana_Júlia_de_Freitas_Carrijo_0948153683101920.xml
Ana_Manuela_Arantes_Costa_1876526830511711.xml
Ana_Maria_de_Morais_4109433138001126.xml
Ana_Rosalva_Osuna_Zamora_4643049391898282.xml
Ana_Terra_Curado_da_Rocha_0060083324284655.xml
André_Almeida_Nunes_2136652274594532.xml
André_Boaratti_1579009638921987.xml
André_Roberto_Custódio_Neves_8630790698956499.xml
Andréa_Pereira_dos_Santos_9315618025567235

In [ ]:
def normalizar_texto(texto):
    """
    normaliza textos para facilitar a comparação e identificação
    de campos dentro do xml.
    """

    if texto is None:
        return ""

    texto = str(texto).strip()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    ).encode(
        "ascii",
        "ignore"
    ).decode(
        "ascii"
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip().lower()

In [ ]:
def extrair_informacoes_basicas_xml(caminho_xml):
    """
    extrai as informações básicas do currículo lattes
    a partir da estrutura xml utilizada pelo projeto.
    """

    try:
        arvore = ET.parse(caminho_xml)
        raiz = arvore.getroot()

    except Exception as erro:

        return {
            "arquivo_xml": os.path.basename(caminho_xml),
            "caminho_xml": caminho_xml,
            "nome": "",
            "id_lattes": "",
            "orcid": "",
            "linkedin": "",
            "data_atualizacao": "",
            "data_extracao": "",
            "resumo_profissional": "",
            "status_xml": "erro",
            "erro_xml": str(erro)
        }

    # --------------------------------------------------
    # informações do elemento raiz
    # --------------------------------------------------

    data_atualizacao = (
        raiz.attrib.get(
            "DATA-ATUALIZACAO",
            ""
        )
    )

    data_extracao = (
        raiz.attrib.get(
            "DATA-EXTRACAO",
            ""
        )
    )

    # --------------------------------------------------
    # identificação
    # --------------------------------------------------

    identificacao = raiz.find(
        "IDENTIFICACAO"
    )

    nome = ""
    id_lattes = ""

    if identificacao is not None:

        elemento_nome = identificacao.find(
            "NOME"
        )

        if elemento_nome is not None:
            nome = (
                elemento_nome.text or ""
            ).strip()

        elemento_id = identificacao.find(
            "ID-LATTES"
        )

        if elemento_id is not None:
            id_lattes = (
                elemento_id.text or ""
            ).strip()

    # --------------------------------------------------
    # links e redes
    # --------------------------------------------------

    orcid = ""
    linkedin = ""

    links_redes = raiz.find(
        "LINKS-E-REDES"
    )

    if links_redes is not None:

        for link in links_redes.findall(
            "LINK"
        ):

            tipo = (
                link.attrib.get(
                    "TIPO",
                    ""
                )
                .strip()
                .upper()
            )

            valor = (
                link.text or ""
            ).strip()

            if tipo == "ORCID":

                if valor.lower() not in [
                    "",
                    "não encontrado",
                    "nao encontrado"
                ]:

                    orcid = valor

            elif tipo == "LINKEDIN":

                if valor.lower() not in [
                    "",
                    "não encontrado",
                    "nao encontrado"
                ]:

                    linkedin = valor

    # --------------------------------------------------
    # resumo profissional
    # --------------------------------------------------

    resumo_profissional = ""

    resumo = raiz.find(
        "RESUMO-PROFISSIONAL"
    )

    if resumo is not None:

        texto = resumo.find(
            "TEXTO"
        )

        if texto is not None:

            resumo_profissional = (
                texto.text or ""
            ).strip()

    # --------------------------------------------------
    # resultado
    # --------------------------------------------------

    return {

        "arquivo_xml":
            os.path.basename(
                caminho_xml
            ),

        "caminho_xml":
            caminho_xml,

        "nome":
            nome,

        "id_lattes":
            id_lattes,

        "orcid":
            orcid,

        "linkedin":
            linkedin,

        "data_atualizacao":
            data_atualizacao,

        "data_extracao":
            data_extracao,

        "resumo_profissional":
            resumo_profissional,

        "status_xml":
            "ok",

        "erro_xml":
            ""
    }

## 2. teste de leitura de um currículo

antes de iniciar o processamento de todos os currículos, será realizado um teste com um único arquivo xml.

o objetivo é verificar se o notebook consegue interpretar corretamente a estrutura do currículo lattes e extrair as informações básicas do egresso.

nesta etapa serão verificadas informações como:

- nome;
- id lattes;
- orcid;
- linkedin;
- data de atualização do currículo;
- data de extração;
- resumo profissional.

o teste também permite identificar se o currículo possui um orcid disponível para a consulta posterior no openalex.

In [ ]:
# selecionar o primeiro xml encontrado para teste

arquivo_teste = arquivos_xml[0]

print("arquivo selecionado para teste:")
print(os.path.basename(arquivo_teste))

arquivo selecionado para teste:
Abdul_Pedro_Manuel_Muchingeca_9666992319712718.xml


In [ ]:
def extrair_informacoes_basicas_xml(caminho_xml):
    """
    extrai informações básicas do currículo lattes
    utilizando a estrutura dos arquivos xml do projeto.
    """

    try:
        arvore = ET.parse(caminho_xml)
        raiz = arvore.getroot()

    except Exception as erro:

        return {
            "arquivo_xml": os.path.basename(caminho_xml),
            "caminho_xml": caminho_xml,
            "nome": "",
            "id_lattes": "",
            "orcid": "",
            "linkedin": "",
            "data_atualizacao": "",
            "data_extracao": "",
            "resumo_profissional": "",
            "status_xml": "erro",
            "erro_xml": str(erro)
        }

    # informações da raiz do currículo

    data_atualizacao = raiz.attrib.get(
        "DATA-ATUALIZACAO",
        ""
    )

    data_extracao = raiz.attrib.get(
        "DATA-EXTRACAO",
        ""
    )

    # identificação

    identificacao = raiz.find(
        "IDENTIFICACAO"
    )

    nome = ""
    id_lattes = ""

    if identificacao is not None:

        elemento_nome = identificacao.find(
            "NOME"
        )

        if elemento_nome is not None:

            nome = (
                elemento_nome.text or ""
            ).strip()

        elemento_id = identificacao.find(
            "ID-LATTES"
        )

        if elemento_id is not None:

            id_lattes = (
                elemento_id.text or ""
            ).strip()

    # links e redes

    orcid = ""
    linkedin = ""

    links_redes = raiz.find(
        "LINKS-E-REDES"
    )

    if links_redes is not None:

        for link in links_redes.findall(
            "LINK"
        ):

            tipo = (
                link.attrib.get(
                    "TIPO",
                    ""
                )
                .strip()
                .upper()
            )

            valor = (
                link.text or ""
            ).strip()

            if tipo == "ORCID":

                if valor.lower() not in [
                    "",
                    "não encontrado",
                    "nao encontrado"
                ]:

                    orcid = valor

            elif tipo == "LINKEDIN":

                if valor.lower() not in [
                    "",
                    "não encontrado",
                    "nao encontrado"
                ]:

                    linkedin = valor

    # resumo profissional

    resumo_profissional = ""

    resumo = raiz.find(
        "RESUMO-PROFISSIONAL"
    )

    if resumo is not None:

        texto = resumo.find(
            "TEXTO"
        )

        if texto is not None:

            resumo_profissional = (
                texto.text or ""
            ).strip()

    return {

        "arquivo_xml":
            os.path.basename(
                caminho_xml
            ),

        "caminho_xml":
            caminho_xml,

        "nome":
            nome,

        "id_lattes":
            id_lattes,

        "orcid":
            orcid,

        "linkedin":
            linkedin,

        "data_atualizacao":
            data_atualizacao,

        "data_extracao":
            data_extracao,

        "resumo_profissional":
            resumo_profissional,

        "status_xml":
            "ok",

        "erro_xml":
            ""
    }

In [ ]:
# executar a leitura do currículo selecionado

dados_teste = extrair_informacoes_basicas_xml(
    arquivo_teste
)

print("resultado da leitura")
print("=" * 60)

print(
    f"arquivo: {dados_teste['arquivo_xml']}"
)

print(
    f"nome: {dados_teste['nome']}"
)

print(
    f"id lattes: {dados_teste['id_lattes']}"
)

print(
    f"orcid: {dados_teste['orcid'] or 'não encontrado'}"
)

print(
    f"linkedin: {dados_teste['linkedin'] or 'não encontrado'}"
)

print(
    f"data de atualização: "
    f"{dados_teste['data_atualizacao']}"
)

print(
    f"data de extração: "
    f"{dados_teste['data_extracao']}"
)

print()

print("resumo profissional:")
print(
    dados_teste["resumo_profissional"]
)

print()

print(
    f"status: {dados_teste['status_xml']}"
)

if dados_teste["erro_xml"]:

    print(
        f"erro: {dados_teste['erro_xml']}"
    )

resultado da leitura
arquivo: Abdul_Pedro_Manuel_Muchingeca_9666992319712718.xml
nome: Abdul Pedro Manuel Muchingeca
id lattes: 9666992319712718
orcid: não encontrado
linkedin: não encontrado
data de atualização: 28/10/2019
data de extração: 20/08/2026 19:55:35

resumo profissional:
Possui graduação em Comunicação Social - Publicidade e Propaganda pela Universidade Federal de Goiás (2014) e Licenciatura em Psicologia pelo INSTITUTO SUPERIOR DE CIÊNCIAS DA EDUCAÇÃO (2010). Mestrado em Comunicação Social, linha de pesquisa Mídia e Cultura. Tem experiência na área de Comunicação, com ênfase em Comunicação

status: ok


In [ ]:
# visualizar o resultado do teste em formato de tabela

df_teste = pd.DataFrame(
    [dados_teste]
)

display(
    df_teste
)

,arquivo_xml,caminho_xml,nome,id_lattes,orcid,linkedin,data_atualizacao,data_extracao,resumo_profissional,status_xml,erro_xml
0,Abdul_Pedro_Manuel_Muchingeca_9666992319712718...,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,Abdul Pedro Manuel Muchingeca,9666992319712718,,,28/10/2019,20/08/2026 19:55:35,Possui graduação em Comunicação Social - Publi...,ok,


## 3. processamento dos currículos lattes

após validar a leitura de um currículo, o notebook irá percorrer todos os arquivos xml disponíveis na pasta dos currículos lattes.

cada arquivo será processado individualmente.

as informações básicas identificadas em cada currículo serão armazenadas em uma tabela para controle do processamento.

nesta etapa ainda não será realizada a consulta ao openalex. primeiro será criada uma base de controle com as informações encontradas nos arquivos xml.

In [ ]:
# processar todos os currículos lattes

resultados_lattes = []

for arquivo_xml in tqdm(
    arquivos_xml,
    desc="lendo currículos lattes"
):

    resultado = extrair_informacoes_basicas_xml(
        arquivo_xml
    )

    resultados_lattes.append(
        resultado
    )

df_lattes = pd.DataFrame(
    resultados_lattes
)

print(
    f"currículos processados: {len(df_lattes)}"
)

display(
    df_lattes
)

lendo currículos lattes:   0%|          | 0/271 [00:00<?, ?it/s]

currículos processados: 271


,arquivo_xml,caminho_xml,nome,id_lattes,orcid,linkedin,data_atualizacao,data_extracao,resumo_profissional,status_xml,erro_xml
0,Abdul_Pedro_Manuel_Muchingeca_9666992319712718...,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,Abdul Pedro Manuel Muchingeca,9666992319712718,,,28/10/2019,20/08/2026 19:55:35,Possui graduação em Comunicação Social - Publi...,ok,
1,Abner_Antônio_Ferreira_0596211008110321.xml,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,Abner Antônio Ferreira,0596211008110321,,,09/02/2026,21/08/2026 13:51:03,Graduado em Administração pela Universidade Es...,ok,
2,Adriana_Teixeira_de_Moraes_3100726985926660.xml,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,Adriana Teixeira de Moraes,3100726985926660,,,25/02/2026,20/08/2026 15:57:41,"Jornalista, Doutoranda em Comunicação , linha...",ok,
3,Adrielly_Campos_e_Almeida_4039893861746107.xml,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,Adrielly Campos e Almeida,4039893861746107,,,15/03/2026,20/08/2026 20:13:05,Mestra em Comunicação Mídia e Cultura pela Uni...,ok,
4,Aldenor_da_Silva_Pimentel_5770288768881926.xml,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,Aldenor da Silva Pimentel,5770288768881926,,,16/01/2026,20/08/2026 15:54:22,Possui graduação em Comunicação Social - Jorna...,ok,
...,...,...,...,...,...,...,...,...,...,...,...
266,William_de_Araújo_Correia_0116797546684695.xml,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,William de Araújo Correia,0116797546684695,,,07/02/2025,21/08/2026 19:28:54,Mestre em Comunicação na linha de pesquisa Míd...,ok,
267,Yasmim_Alves_Fleury_5998175949006985.xml,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,Yasmim Alves Fleury,5998175949006985,,,17/03/2026,20/08/2026 19:51:13,Mestre em Comunicação Social pela Universidade...,ok,
268,Ysabella_de_Medeiros_Portela_3309356144836753.xml,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,Ysabella de Medeiros Portela,3309356144836753,,,19/02/2026,21/08/2026 13:50:41,Doutoranda em Comunicação e Práticas de Consum...,ok,
269,Yuri_Manzi_Giani_3766314156782742.xml,/content/drive/MyDrive/RAGI 2026 2/Extrator de...,Yuri Manzi Giani,3766314156782742,,,04/01/2021,20/08/2026 19:51:26,Mestre em Comunicação pela Universidade Federa...,ok,


In [ ]:
# verificar a disponibilidade de orcid nos currículos

total_curriculos = len(
    df_lattes
)

total_com_orcid = (
    df_lattes["orcid"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

total_sem_orcid = (
    total_curriculos
    - total_com_orcid
)

print(
    f"total de currículos: {total_curriculos}"
)

print(
    f"currículos com orcid: {total_com_orcid}"
)

print(
    f"currículos sem orcid: {total_sem_orcid}"
)

total de currículos: 271
currículos com orcid: 79
currículos sem orcid: 192


In [ ]:
# apresentar as informações principais dos currículos

colunas_controle = [
    "arquivo_xml",
    "nome",
    "id_lattes",
    "orcid",
    "linkedin",
    "data_atualizacao",
    "data_extracao",
    "status_xml"
]

display(
    df_lattes[
        colunas_controle
    ]
)

,arquivo_xml,nome,id_lattes,orcid,linkedin,data_atualizacao,data_extracao,status_xml
0,Abdul_Pedro_Manuel_Muchingeca_9666992319712718...,Abdul Pedro Manuel Muchingeca,9666992319712718,,,28/10/2019,20/08/2026 19:55:35,ok
1,Abner_Antônio_Ferreira_0596211008110321.xml,Abner Antônio Ferreira,0596211008110321,,,09/02/2026,21/08/2026 13:51:03,ok
2,Adriana_Teixeira_de_Moraes_3100726985926660.xml,Adriana Teixeira de Moraes,3100726985926660,,,25/02/2026,20/08/2026 15:57:41,ok
3,Adrielly_Campos_e_Almeida_4039893861746107.xml,Adrielly Campos e Almeida,4039893861746107,,,15/03/2026,20/08/2026 20:13:05,ok
4,Aldenor_da_Silva_Pimentel_5770288768881926.xml,Aldenor da Silva Pimentel,5770288768881926,,,16/01/2026,20/08/2026 15:54:22,ok
...,...,...,...,...,...,...,...,...
266,William_de_Araújo_Correia_0116797546684695.xml,William de Araújo Correia,0116797546684695,,,07/02/2025,21/08/2026 19:28:54,ok
267,Yasmim_Alves_Fleury_5998175949006985.xml,Yasmim Alves Fleury,5998175949006985,,,17/03/2026,20/08/2026 19:51:13,ok
268,Ysabella_de_Medeiros_Portela_3309356144836753.xml,Ysabella de Medeiros Portela,3309356144836753,,,19/02/2026,21/08/2026 13:50:41,ok
269,Yuri_Manzi_Giani_3766314156782742.xml,Yuri Manzi Giani,3766314156782742,,,04/01/2021,20/08/2026 19:51:26,ok


In [ ]:
# separar os currículos de acordo com a existência de orcid

df_com_orcid = df_lattes[
    df_lattes["orcid"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
].copy()

df_sem_orcid = df_lattes[
    df_lattes["orcid"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
].copy()

print(
    f"currículos disponíveis para consulta no openalex: "
    f"{len(df_com_orcid)}"
)

print(
    f"currículos sem orcid: "
    f"{len(df_sem_orcid)}"
)

currículos disponíveis para consulta no openalex: 79
currículos sem orcid: 192


## 4. consulta ao openalex

os currículos que possuem um orcid identificado serão utilizados como entrada para a consulta à api do openalex.

o orcid será utilizado para localizar as obras associadas ao pesquisador.

para cada publicação encontrada serão coletados os metadados disponíveis no openalex, permitindo complementar as informações provenientes do currículo lattes.

os currículos que não possuem orcid não serão consultados nesta etapa e permanecerão registrados na base de controle.

In [ ]:
# configurações da api do openalex

OPENALEX_API = "https://api.openalex.org/works"

# quantidade de publicações retornadas por página
PER_PAGE = 200

# tempo máximo de espera por requisição
TIMEOUT = 30

# quantidade máxima de tentativas em caso de erro
MAX_TENTATIVAS = 3

# intervalo entre consultas de diferentes egressos
INTERVALO_ENTRE_EGRESSOS = 0.5

# limite opcional para testes
# use None para processar todos os egressos
LIMITE_PROCESSAMENTO = None

### consulta das publicações

a função desta etapa recebe o orcid identificado no currículo lattes e realiza a consulta correspondente no openalex.

a consulta utiliza o identificador orcid para localizar as obras associadas ao pesquisador.

para cada publicação encontrada serão extraídas informações bibliográficas, indicadores de citação, autoria, instituições, áreas de conhecimento, acesso aberto, financiamento e ods, quando essas informações estiverem disponíveis na resposta da api.

In [ ]:
def extrair_dados_completos_openalex(orcid):
    """
    consulta as obras de um pesquisador no openalex
    utilizando seu orcid.
    """

    if not orcid:
        return [], "orcid vazio"

    orcid = str(orcid).strip()

    # caso o xml tenha armazenado a url completa
    orcid = re.sub(
        r"^https?://orcid\.org/",
        "",
        orcid,
        flags=re.IGNORECASE
    )

    orcid = orcid.strip().upper()

    orcid_url = f"https://orcid.org/{orcid}"

    trabalhos = []

    page = 1

    headers = {
        "User-Agent": "RAGI-OpenAlex/1.0"
    }

    while True:

        params = {
            "filter": f"author.orcid:{orcid_url}",
            "per-page": PER_PAGE,
            "page": page
        }

        data = None
        ultimo_erro = ""

        for tentativa in range(
            1,
            MAX_TENTATIVAS + 1
        ):

            try:

                response = requests.get(
                    OPENALEX_API,
                    params=params,
                    headers=headers,
                    timeout=TIMEOUT
                )

                if response.status_code == 200:

                    data = response.json()

                    break

                ultimo_erro = (
                    f"HTTP {response.status_code}"
                )

                if response.status_code in [
                    429,
                    500,
                    502,
                    503,
                    504
                ]:

                    time.sleep(
                        2 * tentativa
                    )

                    continue

                return trabalhos, ultimo_erro

            except requests.RequestException as erro:

                ultimo_erro = (
                    f"{type(erro).__name__}: {erro}"
                )

                time.sleep(
                    2 * tentativa
                )

        if data is None:

            return trabalhos, ultimo_erro

        resultados = data.get(
            "results",
            []
        )

        if not resultados:
            break

        for item in resultados:

            # ------------------------------------------
            # localização principal
            # ------------------------------------------

            primary_location = (
                item.get(
                    "primary_location"
                )
                or {}
            )

            source = (
                primary_location.get(
                    "source"
                )
                or {}
            )

            # ------------------------------------------
            # acesso aberto
            # ------------------------------------------

            open_access = (
                item.get(
                    "open_access"
                )
                or {}
            )

            # ------------------------------------------
            # autores e instituições
            # ------------------------------------------

            authorships = (
                item.get(
                    "authorships"
                )
                or []
            )

            coautores = []

            instituicoes = set()

            paises = set()

            for authorship in authorships:

                author = (
                    authorship.get(
                        "author"
                    )
                    or {}
                )

                nome_autor = (
                    author.get(
                        "display_name"
                    )
                )

                if nome_autor:

                    coautores.append(
                        nome_autor
                    )

                for instituicao in (
                    authorship.get(
                        "institutions"
                    )
                    or []
                ):

                    nome_instituicao = (
                        instituicao.get(
                            "display_name"
                        )
                    )

                    pais = (
                        instituicao.get(
                            "country_code"
                        )
                    )

                    if nome_instituicao:

                        instituicoes.add(
                            nome_instituicao
                        )

                    if pais:

                        paises.add(
                            pais
                        )

            # ------------------------------------------
            # tópico principal
            # ------------------------------------------

            primary_topic = (
                item.get(
                    "primary_topic"
                )
                or {}
            )

            subfield = (
                primary_topic.get(
                    "subfield"
                )
                or {}
            )

            field = (
                primary_topic.get(
                    "field"
                )
                or {}
            )

            domain = (
                primary_topic.get(
                    "domain"
                )
                or {}
            )

            # ------------------------------------------
            # conceitos
            # ------------------------------------------

            concepts = (
                item.get(
                    "concepts"
                )
                or []
            )

            concepts_sorted = sorted(
                concepts,
                key=lambda x: (
                    x.get(
                        "score",
                        0
                    )
                    or 0
                ),
                reverse=True
            )

            top_concepts = [
                conceito.get(
                    "display_name"
                )
                for conceito in concepts_sorted[:5]
                if conceito.get(
                    "display_name"
                )
            ]

            # ------------------------------------------
            # histórico de citações
            # ------------------------------------------

            counts_by_year = (
                item.get(
                    "counts_by_year"
                )
                or []
            )

            historico_citacoes = []

            for registro in counts_by_year:

                ano = registro.get(
                    "year"
                )

                citacoes = registro.get(
                    "cited_by_count"
                )

                if ano is not None:

                    historico_citacoes.append(
                        f"{ano}:{citacoes}"
                    )

            # ------------------------------------------
            # financiadores
            # ------------------------------------------

            grants = (
                item.get(
                    "grants"
                )
                or []
            )

            financiadores = sorted(
                {
                    grant.get(
                        "funder_display_name"
                    )
                    for grant in grants
                    if grant.get(
                        "funder_display_name"
                    )
                }
            )

            # ------------------------------------------
            # ods
            # ------------------------------------------

            sdgs = (
                item.get(
                    "sustainable_development_goals"
                )
                or []
            )

            ods_lista = [
                sdg.get(
                    "display_name"
                )
                for sdg in sdgs
                if sdg.get(
                    "display_name"
                )
            ]

            # ------------------------------------------
            # registro da publicação
            # ------------------------------------------

            trabalhos.append({

                "ID_OpenAlex":
                    item.get("id"),

                "DOI":
                    item.get("doi"),

                "Titulo_Trabalho":
                    item.get("title"),

                "Ano_Publicacao":
                    item.get("publication_year"),

                "Data_Exata_Publicacao":
                    item.get("publication_date"),

                "Tipo_Documento":
                    item.get("type"),

                "Idioma":
                    item.get("language"),

                "Citacoes_Totais":
                    item.get(
                        "cited_by_count",
                        0
                    ),

                "Historico_Citacoes_Ano":
                    " | ".join(
                        historico_citacoes
                    ),

                "Qtd_Referencias_Citadas":
                    len(
                        item.get(
                            "referenced_works"
                        )
                        or []
                    ),

                "Qtd_Obras_Relacionadas":
                    len(
                        item.get(
                            "related_works"
                        )
                        or []
                    ),

                "Acesso_Aberto":
                    (
                        "Sim"
                        if open_access.get(
                            "is_oa"
                        )
                        else "Não"
                    ),

                "Status_Acesso_Aberto":
                    open_access.get(
                        "oa_status"
                    ),

                "URL_PDF_Aberto":
                    primary_location.get(
                        "pdf_url"
                    ),

                "URL_Pagina_Publicacao":
                    primary_location.get(
                        "landing_page_url"
                    ),

                "Nome_Veiculo":
                    source.get(
                        "display_name"
                    ),

                "Tipo_Veiculo":
                    source.get(
                        "type"
                    ),

                "Editora_Publisher":
                    source.get(
                        "host_organization_name"
                    ),

                "ISSN":
                    source.get(
                        "issn_l"
                    ),

                "Total_Autores":
                    len(authorships),

                "Lista_Coautores":
                    " ; ".join(
                        coautores
                    ),

                "Instituicoes_Filiadas":
                    " ; ".join(
                        sorted(
                            instituicoes
                        )
                    ),

                "Paises_Instituicoes":
                    " ; ".join(
                        sorted(
                            paises
                        )
                    ),

                "Topico_Principal":
                    primary_topic.get(
                        "display_name"
                    ),

                "Subcampo_Conhecimento":
                    subfield.get(
                        "display_name"
                    ),

                "Campo_Conhecimento":
                    field.get(
                        "display_name"
                    ),

                "Grande_Area_Conhecimento":
                    domain.get(
                        "display_name"
                    ),

                "Palavras_Chave_Conceitos":
                    " | ".join(
                        top_concepts
                    ),

                "Agencias_Financiadoras":
                    " ; ".join(
                        financiadores
                    ),

                "ODS_ONU_Associados":
                    " ; ".join(
                        ods_lista
                    )
            })

        # se retornou menos que o limite,
        # não há outra página
        if len(resultados) < PER_PAGE:
            break

        page += 1

    return trabalhos, ""

### teste da consulta ao openalex

antes de processar todos os egressos, será realizada uma consulta utilizando o primeiro currículo que possui orcid.

o objetivo é confirmar que:

- o orcid foi extraído corretamente do xml;
- a consulta ao openalex está funcionando;
- as publicações estão sendo retornadas;
- os principais campos estão sendo preenchidos.

caso o primeiro currículo disponível não possua orcid, o notebook selecionará automaticamente outro currículo que possua o identificador.

In [ ]:
# verificar se existe algum currículo com orcid

if df_com_orcid.empty:

    print(
        "nenhum currículo com orcid foi encontrado."
    )

else:

    linha_teste = df_com_orcid.iloc[0]

    nome_teste = linha_teste[
        "nome"
    ]

    orcid_teste = linha_teste[
        "orcid"
    ]

    arquivo_teste_openalex = linha_teste[
        "arquivo_xml"
    ]

    print(
        "currículo selecionado:"
    )

    print(
        f"nome: {nome_teste}"
    )

    print(
        f"orcid: {orcid_teste}"
    )

    print(
        f"arquivo: {arquivo_teste_openalex}"
    )

currículo selecionado:
nome: Alessandra Rodrigues Oliveira
orcid: https://orcid.org/0000-0001-8826-5510
arquivo: Alessandra_Rodrigues_Oliveira_9237912840073975.xml


In [ ]:
# consultar o openalex utilizando o orcid do egresso selecionado

if not df_com_orcid.empty:

    publicacoes_teste, erro_teste = (
        extrair_dados_completos_openalex(
            orcid_teste
        )
    )

    print(
        f"publicações encontradas: "
        f"{len(publicacoes_teste)}"
    )

    print()

    if erro_teste:

        print(
            "erro na consulta:"
        )

        print(
            erro_teste
        )

    else:

        print(
            "consulta realizada com sucesso."
        )

publicações encontradas: 0

consulta realizada com sucesso.


In [ ]:
if publicacoes_teste:

    df_publicacoes_teste = pd.DataFrame(
        publicacoes_teste
    )

    colunas_teste = [
        "ID_OpenAlex",
        "DOI",
        "Titulo_Trabalho",
        "Ano_Publicacao",
        "Citacoes_Totais",
        "Nome_Veiculo",
        "Lista_Coautores",
        "Topico_Principal"
    ]

    colunas_teste = [
        coluna
        for coluna in colunas_teste
        if coluna in df_publicacoes_teste.columns
    ]

    display(
        df_publicacoes_teste[
            colunas_teste
        ]
    )

else:

    print(
        "nenhuma publicação foi encontrada "
        "para o orcid testado."
    )

nenhuma publicação foi encontrada para o orcid testado.


## 5. processamento completo dos egressos

após validar a consulta individual, o notebook irá consultar o openalex para todos os currículos que possuem orcid.

cada publicação será associada ao egresso de origem por meio do nome, orcid e arquivo xml utilizado no processamento.

também serão calculados indicadores consolidados por egresso, como:

- quantidade de publicações;
- quantidade total de citações;
- quantidade de publicações em acesso aberto;
- países das instituições de colaboração;
- status da consulta.

eventuais erros serão registrados separadamente para permitir a conferência posterior sem interromper o processamento dos demais egressos.

In [ ]:
# listas para armazenar os resultados

todas_publicacoes = []

resumo_egressos = []

erros_processamento = []

# selecionar os egressos que serão processados

df_processamento = df_com_orcid.copy()

if LIMITE_PROCESSAMENTO is not None:

    df_processamento = (
        df_processamento
        .head(LIMITE_PROCESSAMENTO)
        .copy()
    )

print(
    f"egressos que serão consultados: "
    f"{len(df_processamento)}"
)

print()

for _, linha in tqdm(
    df_processamento.iterrows(),
    total=len(df_processamento),
    desc="consultando openalex"
):

    nome = linha["nome"]

    orcid = linha["orcid"]

    arquivo_xml = linha["arquivo_xml"]

    try:

        publicacoes, erro = (
            extrair_dados_completos_openalex(
                orcid
            )
        )

    except Exception as erro_execucao:

        publicacoes = []

        erro = (
            f"{type(erro_execucao).__name__}: "
            f"{erro_execucao}"
        )

    # adicionar os dados do egresso
    # a cada publicação encontrada

    for publicacao in publicacoes:

        registro = {

            "Discente_Lattes":
                nome,

            "ID_Lattes":
                linha["id_lattes"],

            "ORCID_Lattes":
                orcid,

            "Arquivo_XML_Lattes":
                arquivo_xml,

            "Data_Atualizacao_Lattes":
                linha["data_atualizacao"],

            **publicacao
        }

        todas_publicacoes.append(
            registro
        )

    # ------------------------------------------
    # países das instituições
    # ------------------------------------------

    paises = set()

    for publicacao in publicacoes:

        valor = (
            publicacao.get(
                "Paises_Instituicoes",
                ""
            )
            or ""
        )

        for pais in valor.split(" ; "):

            pais = pais.strip()

            if pais:

                paises.add(
                    pais
                )

    # ------------------------------------------
    # indicadores do egresso
    # ------------------------------------------

    total_citacoes = sum(
        int(
            publicacao.get(
                "Citacoes_Totais",
                0
            )
            or 0
        )
        for publicacao in publicacoes
    )

    total_acesso_aberto = sum(
        1
        for publicacao in publicacoes
        if publicacao.get(
            "Acesso_Aberto"
        ) == "Sim"
    )

    resumo_egressos.append({

        "Discente":
            nome,

        "ID_Lattes":
            linha["id_lattes"],

        "ORCID":
            orcid,

        "Arquivo_XML":
            arquivo_xml,

        "Data_Atualizacao_Lattes":
            linha["data_atualizacao"],

        "Total_Publicacoes_OpenAlex":
            len(publicacoes),

        "Total_Citacoes_Acumuladas":
            total_citacoes,

        "Qtd_Publicacoes_Acesso_Aberto":
            total_acesso_aberto,

        "Paises_Colaboracao":
            " ; ".join(
                sorted(paises)
            ),

        "Status_OpenAlex":
            (
                "concluído"
                if not erro
                else "erro"
            ),

        "Erro_OpenAlex":
            erro,

        "Data_Consulta_OpenAlex":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    })

    # ------------------------------------------
    # registrar erros
    # ------------------------------------------

    if erro:

        erros_processamento.append({

            "Discente":
                nome,

            "ID_Lattes":
                linha["id_lattes"],

            "ORCID":
                orcid,

            "Arquivo_XML":
                arquivo_xml,

            "Erro":
                erro
        })

    # intervalo entre egressos

    time.sleep(
        INTERVALO_ENTRE_EGRESSOS
    )

print()

print(
    "processamento concluído."
)

egressos que serão consultados: 79



consultando openalex:   0%|          | 0/79 [00:00<?, ?it/s]


processamento concluído.


In [ ]:
# transformar os resultados em dataframes

df_pubs = pd.DataFrame(
    todas_publicacoes
)

df_resumo = pd.DataFrame(
    resumo_egressos
)

df_erros = pd.DataFrame(
    erros_processamento
)

print(
    f"egressos processados: "
    f"{len(df_resumo)}"
)

print(
    f"publicações encontradas: "
    f"{len(df_pubs)}"
)

print(
    f"consultas com erro: "
    f"{len(df_erros)}"
)

egressos processados: 79
publicações encontradas: 672
consultas com erro: 0


## 6. conferência dos resultados

nesta etapa serão apresentados os resultados obtidos durante o processamento.

a base será separada em diferentes visões para facilitar a conferência:

- resumo dos indicadores por egresso;
- publicações detalhadas;
- erros encontrados nas consultas;
- currículos que não possuem orcid.

essa separação permite identificar rapidamente problemas de identificação ou de consulta antes da utilização dos dados nas próximas etapas do projeto.

In [ ]:
if not df_resumo.empty:

    display(
        df_resumo
    )

,Discente,ID_Lattes,ORCID,Arquivo_XML,Data_Atualizacao_Lattes,Total_Publicacoes_OpenAlex,Total_Citacoes_Acumuladas,Qtd_Publicacoes_Acesso_Aberto,Paises_Colaboracao,Status_OpenAlex,Erro_OpenAlex,Data_Consulta_OpenAlex
0,Alessandra Rodrigues Oliveira,9237912840073975,https://orcid.org/0000-0001-8826-5510,Alessandra_Rodrigues_Oliveira_9237912840073975...,28/08/2023,0,0,0,,concluído,,2026-08-26 18:07:20
1,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,Ana_Júlia_de_Freitas_Carrijo_0948153683101920...,30/07/2026,13,7,12,BR,concluído,,2026-08-26 18:07:21
2,Andréa Pereira dos Santos,9315618025567235,https://orcid.org/0000-0001-5410-5500,Andréa_Pereira_dos_Santos_9315618025567235.xml,25/02/2026,56,24,36,BR ; ES ; ID ; US,concluído,,2026-08-26 18:07:22
3,Antônio Guilherme de Lima Santos,9779435301297884,https://orcid.org/0000-0002-4218-5627,Antônio_Guilherme_de_Lima_Santos_977943530129...,14/08/2026,0,0,0,,concluído,,2026-08-26 18:07:22
4,Augusto Flamaryon Cecchin Bozz,9837974802580178,https://orcid.org/0000-0002-5723-2158,Augusto_Flamaryon_Cecchin_Bozz_983797480258017...,11/08/2026,5,3,5,BR,concluído,,2026-08-26 18:07:23
...,...,...,...,...,...,...,...,...,...,...,...,...
74,Thiago Cardoso Franco,7735628705551412,https://orcid.org/0000-0002-1774-4706,Thiago_Cardoso_Franco_7735628705551412.xml,27/03/2026,33,17,24,BR ; CA ; FR ; MX ; PT ; TD ; US,concluído,,2026-08-26 18:08:19
75,Thiago da Silva Rabelo,6159420155328823,https://orcid.org/0009-0001-5913-7015,Thiago_da_Silva_Rabelo_6159420155328823.xml,29/06/2026,4,0,2,BR,concluído,,2026-08-26 18:08:20
76,Tony Willian Boita,9179927304604562,https://orcid.org/0000-0003-3780-2157,Tony_Willian_Boita_9179927304604562.xml,13/04/2026,32,36,24,BE ; BR ; CA ; CZ ; EC ; ES ; FR ; MX ; PT ; US,concluído,,2026-08-26 18:08:21
77,Vinicius Luiz Tondolo,9115597246274622,https://orcid.org/0000-0001-5716-926X,Vinicius_Luiz_Tondolo_9115597246274622.xml,11/08/2025,0,0,0,,concluído,,2026-08-26 18:08:22


In [ ]:
if not df_pubs.empty:

    colunas_visualizacao = [
        "Discente_Lattes",
        "ID_Lattes",
        "ORCID_Lattes",
        "Titulo_Trabalho",
        "Ano_Publicacao",
        "DOI",
        "Citacoes_Totais",
        "Acesso_Aberto",
        "Nome_Veiculo",
        "Total_Autores",
        "Topico_Principal"
    ]

    colunas_existentes = [
        coluna
        for coluna in colunas_visualizacao
        if coluna in df_pubs.columns
    ]

    display(
        df_pubs[
            colunas_existentes
        ].head(30)
    )
else:

    print(
        "nenhuma publicação foi encontrada."
    )

,Discente_Lattes,ID_Lattes,ORCID_Lattes,Titulo_Trabalho,Ano_Publicacao,DOI,Citacoes_Totais,Acesso_Aberto,Nome_Veiculo,Total_Autores,Topico_Principal
0,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,Algoritmos e cultura digital,2024,https://doi.org/10.15448/1980-3729.2024.1.44941,3,Sim,Revista FAMECOS,2,Media and Digital Communication
1,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,Rumo a uma abordagem sociocultural de tecnolog...,2023,https://doi.org/10.34019/1981-4070.2023.v17.41094,2,Sim,Lumina,2,Education and Digital Technologies
2,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,O que adolescentes pensam sobre o compartilham...,2021,https://doi.org/10.22409/rmc.v15i1.43146,1,Sim,Revista Mídia e Cotidiano,2,Social Media and Politics
3,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,Padrões de performance no Instagram na experiê...,2021,https://doi.org/10.29146/ecopos.v24i3.27666,1,Sim,Revista ECO-Pós,2,Social Media and Politics
4,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,A identificação do público infantil com youtub...,2019,https://doi.org/10.20435/multi.v24i56.2144,0,Sim,Multitemas,2,Education and Digital Technologies
5,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,INFÂNCIA E YOUTUBE,2019,None,0,Sim,DOAJ (DOAJ: Directory of Open Access Journals),2,"Cultural, Media, and Literary Studies"
6,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,O MÉTODO DA TOTALIDADE E OS PARADIGMAS ACADÊMI...,2020,https://doi.org/10.5216/teri.v10i1.65199,0,Sim,Revista Terceiro Incluído,2,Media and Communication Studies
7,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,Incomunicabilidade na internet:,2021,https://doi.org/10.30962/ec.2538,0,Sim,E-Compós,3,Social Media and Politics
8,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,Articulações teóricometodológicas em uma exper...,2021,https://doi.org/10.19132/1807-8583202253.111888,0,Sim,Intexto,2,"Cultural, Media, and Literary Studies"
9,Ana Júlia de Freitas Carrijo,0948153683101920,https://orcid.org/0000-0001-8372-1003,Paola Ricaurte-Quijano: tecnologias digitais a...,2024,https://doi.org/10.22409/rmc.v18i3.63444,0,Sim,Revista Mídia e Cotidiano,2,Media and Digital Communication


In [ ]:
# visualizar os currículos que não possuem orcid

colunas_sem_orcid = [
    "arquivo_xml",
    "nome",
    "id_lattes",
    "data_atualizacao",
    "status_xml",
    "erro_xml"
]

colunas_sem_orcid = [
    coluna
    for coluna in colunas_sem_orcid
    if coluna in df_sem_orcid.columns
]

display(
    df_sem_orcid[
        colunas_sem_orcid
    ]
)

,arquivo_xml,nome,id_lattes,data_atualizacao,status_xml,erro_xml
0,Abdul_Pedro_Manuel_Muchingeca_9666992319712718...,Abdul Pedro Manuel Muchingeca,9666992319712718,28/10/2019,ok,
1,Abner_Antônio_Ferreira_0596211008110321.xml,Abner Antônio Ferreira,0596211008110321,09/02/2026,ok,
2,Adriana_Teixeira_de_Moraes_3100726985926660.xml,Adriana Teixeira de Moraes,3100726985926660,25/02/2026,ok,
3,Adrielly_Campos_e_Almeida_4039893861746107.xml,Adrielly Campos e Almeida,4039893861746107,15/03/2026,ok,
4,Aldenor_da_Silva_Pimentel_5770288768881926.xml,Aldenor da Silva Pimentel,5770288768881926,16/01/2026,ok,
...,...,...,...,...,...,...
266,William_de_Araújo_Correia_0116797546684695.xml,William de Araújo Correia,0116797546684695,07/02/2025,ok,
267,Yasmim_Alves_Fleury_5998175949006985.xml,Yasmim Alves Fleury,5998175949006985,17/03/2026,ok,
268,Ysabella_de_Medeiros_Portela_3309356144836753.xml,Ysabella de Medeiros Portela,3309356144836753,19/02/2026,ok,
269,Yuri_Manzi_Giani_3766314156782742.xml,Yuri Manzi Giani,3766314156782742,04/01/2021,ok,


In [ ]:
# visualizar erros ocorridos durante as consultas

if df_erros.empty:

    print(
        "nenhum erro foi registrado."
    )

else:

    display(
        df_erros
    )

nenhum erro foi registrado.


## 7. exportação dos resultados

ao final do processamento, os resultados serão organizados em um arquivo excel.

o arquivo terá abas separadas para manter a estrutura dos dados e facilitar sua utilização nas próximas etapas do projeto.

as principais abas serão:

- **resumo de indicadores:** informações consolidadas por egresso;
- **publicações detalhadas:** uma linha para cada publicação encontrada no openalex;
- **erros openalex:** registros de consultas que apresentaram problemas;
- **currículos sem orcid:** currículos para os quais não foi identificado um orcid;
- **controle lattes:** informações básicas extraídas diretamente dos arquivos xml.

In [ ]:
# criar diretório de saída da estratégia 1

DIRETORIO_SAIDA = (
    "/content/drive/MyDrive/RAGI 2026 2/"
    "Enriquecimento de dados - OPENALEX/"
    "Estratégia 1"
)

os.makedirs(
    DIRETORIO_SAIDA,
    exist_ok=True
)

ARQUIVO_SAIDA = os.path.join(
    DIRETORIO_SAIDA,
    "publicacoes_egressos_openalex.xlsx"
)

print("diretório de saída:")
print(DIRETORIO_SAIDA)

print()

print("arquivo final:")
print(ARQUIVO_SAIDA)

print()

if os.path.isdir(DIRETORIO_SAIDA):

    print("✓ pasta de saída pronta.")

else:

    print("✗ não foi possível criar a pasta.")

diretório de saída:
/content/drive/MyDrive/RAGI 2026 2/Enriquecimento de dados - OPENALEX/Estratégia 1

arquivo final:
/content/drive/MyDrive/RAGI 2026 2/Enriquecimento de dados - OPENALEX/Estratégia 1/publicacoes_egressos_openalex.xlsx

✓ pasta de saída pronta.


In [ ]:
# salvar os resultados em excel

with pd.ExcelWriter(
    ARQUIVO_SAIDA,
    engine="openpyxl"
) as writer:

    df_resumo.to_excel(
        writer,
        sheet_name="Resumo de Indicadores",
        index=False
    )

    df_pubs.to_excel(
        writer,
        sheet_name="Publicações Detalhadas",
        index=False
    )

    df_erros.to_excel(
        writer,
        sheet_name="Erros OpenAlex",
        index=False
    )

    df_sem_orcid.to_excel(
        writer,
        sheet_name="Currículos sem ORCID",
        index=False
    )

    df_lattes.to_excel(
        writer,
        sheet_name="Controle Lattes",
        index=False
    )

print(
    "arquivo salvo com sucesso!"
)

print()

print(
    ARQUIVO_SAIDA
)

arquivo salvo com sucesso!

/content/drive/MyDrive/RAGI 2026 2/Enriquecimento de dados - OPENALEX/Estratégia 1/publicacoes_egressos_openalex.xlsx


In [ ]:
print("=" * 60)
print("resumo da execução")
print("=" * 60)

print(
    f"currículos xml encontrados: "
    f"{len(arquivos_xml)}"
)

print(
    f"currículos processados: "
    f"{len(df_lattes)}"
)

print(
    f"currículos com orcid: "
    f"{len(df_com_orcid)}"
)

print(
    f"currículos sem orcid: "
    f"{len(df_sem_orcid)}"
)

print(
    f"egressos consultados no openalex: "
    f"{len(df_resumo)}"
)

print(
    f"publicações encontradas: "
    f"{len(df_pubs)}"
)

print(
    f"consultas com erro: "
    f"{len(df_erros)}"
)

print()

print(
    "arquivo final:"
)

print(
    ARQUIVO_SAIDA
)

print("=" * 60)

resumo da execução
currículos xml encontrados: 271
currículos processados: 271
currículos com orcid: 79
currículos sem orcid: 192
egressos consultados no openalex: 79
publicações encontradas: 672
consultas com erro: 0

arquivo final:
/content/drive/MyDrive/RAGI 2026 2/Enriquecimento de dados - OPENALEX/Estratégia 1/publicacoes_egressos_openalex.xlsx
